# Fase 5 — Evaluación y análisis de complejidad
## Métricas del modelo + análisis costo-comunicación del pipeline completo

**Entrada:** predicciones del MLP (Fase 4) + estadísticas de las fases anteriores  
**Contenido:**
1. Métricas de calidad del clasificador (F1, accuracy, precisión, recall por clase)
2. Matriz de confusión
3. Análisis costo-comunicación de cada fase del pipeline
4. Análisis de complejidad algorítmica y escalabilidad

## 0. Rutas

In [ ]:
import os

PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATOS_PATH   = os.path.join(PROJECT_PATH, "datos")
MODELS_PATH  = os.path.join(PROJECT_PATH, "models")

PREDS_PATH    = os.path.join(DATOS_PATH, "processed", "predicciones")
TFIDF_MIND    = os.path.join(DATOS_PATH, "processed", "tfidf_mind")
DEDUP_PATH    = os.path.join(DATOS_PATH, "dedup")
OUT_EVAL      = os.path.join(PROJECT_PATH, "resultados")

os.makedirs(OUT_EVAL, exist_ok=True)

print(f"PROJECT_PATH → {PROJECT_PATH}")
print(f"Predicciones → {PREDS_PATH}  ({'OK' if os.path.exists(PREDS_PATH) else 'FALTA'})")

## 1. Dependencias

In [ ]:
# Ejecuta solo la primera vez
# !pip install pyspark scikit-learn matplotlib seaborn

## 2. Inicialización de PySpark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Evaluacion-Pipeline")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"PySpark {spark.version} listo.")

## 3. Carga de predicciones

In [ ]:
preds_df = spark.read.parquet(PREDS_PATH)

print(f"Predicciones cargadas: {preds_df.count():,} documentos")
preds_df.show(5, truncate=False)

# Convertir a pandas para métricas con sklearn
preds_pd = preds_df.select("label", "prediccion", "label_idx", "prediccion_idx").toPandas()
print(f"\nClases únicas: {sorted(preds_pd['label'].unique().tolist())}")

## 4. Métricas globales

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

y_true = preds_pd["label_idx"].astype(int)
y_pred = preds_pd["prediccion_idx"].astype(int)

acc       = accuracy_score(y_true, y_pred)
f1_w      = f1_score(y_true, y_pred, average="weighted")
f1_macro  = f1_score(y_true, y_pred, average="macro")
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall    = recall_score(y_true, y_pred, average="weighted", zero_division=0)

print("=" * 45)
print("MÉTRICAS GLOBALES — CLASIFICADOR MLP")
print("=" * 45)
print(f"  Accuracy              : {acc:.4f}")
print(f"  F1 ponderado          : {f1_w:.4f}")
print(f"  F1 macro              : {f1_macro:.4f}")
print(f"  Precisión ponderada   : {precision:.4f}")
print(f"  Recall ponderado      : {recall:.4f}")
print("=" * 45)

## 5. F1 por clase

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

labels = sorted(preds_pd["label"].unique().tolist())

report = classification_report(
    preds_pd["label"],
    preds_pd["prediccion"],
    labels=labels,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T
report_df = report_df[report_df.index.isin(labels)].sort_values("f1-score", ascending=False)

print("F1 por clase (ordenado de mayor a menor):")
print(report_df[["precision", "recall", "f1-score", "support"]].to_string(float_format="{:.4f}".format))

## 6. Matriz de confusión

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(preds_pd["label"], preds_pd["prediccion"], labels=labels)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    ax=ax
)
ax.set_xlabel("Predicción", fontsize=12)
ax.set_ylabel("Real", fontsize=12)
ax.set_title("Matriz de Confusión — Clasificador MLP", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

fig.savefig(os.path.join(OUT_EVAL, "confusion_matrix.png"), dpi=150)
plt.show()
print("Guardada en resultados/confusion_matrix.png")

## 7. Análisis costo-comunicación del pipeline completo

Aplicamos el **modelo costo-comunicación** del curso a cada fase.

Sea $N$ = número de documentos, $L$ = longitud media de documento, $|V|$ = vocabulario,
$K$ = funciones hash MinHash, $p$ = número de particiones, $W$ = parámetros del MLP.

In [ ]:
# Estadísticas reales del pipeline
tfidf_df = spark.read.parquet(TFIDF_MIND)
N        = tfidf_df.count()
N_CC     = 703488   # CC-News procesados (Fase 1)
N_total  = N + N_CC

# Parámetros del pipeline
NUM_FEATURES    = 65536
CAPAS           = [65536, 256, 128, 18]
K_MINHASH       = 5
P_PARTICIONES   = 8
W_MLP           = sum(CAPAS[i]*CAPAS[i+1] + CAPAS[i+1] for i in range(len(CAPAS)-1))

print("ANÁLISIS COSTO-COMUNICACIÓN — PIPELINE COMPLETO")
print("=" * 62)
print(f"{'Fase':<35} {'Cómputo':<18} {'Comunicación'}")
print("-" * 62)
print(f"{'Fase 1 — Ingesta (N docs)':<35} {'O(N)':<18} {'O(N) — 1 lectura'}")
print(f"{'Fase 2 — Map (tokenizar)':<35} {'O(N·L)':<18} {'O(0) — local'}")
print(f"{'Fase 2 — Combiner (stem)':<35} {'O(N·L)':<18} {'O(0) — local'}")
print(f"{'Fase 2 — Reduce IDF (shuffle)':<35} {'O(|V|·p)':<18} {'O(|V|) — costoso'}")
print(f"{'Fase 2 — Índice invertido':<35} {'O(N·L)':<18} {'O(N·L) — groupByKey'}")
print(f"{'Fase 3 — MinHash (firmas)':<35} {'O(N·K·L)':<18} {'O(N·K)'}")
print(f"{'Fase 3 — LSH banding':<35} {'O(N·K)':<18} {'O(N·K) — shuffle'}")
print(f"{'Fase 3 — Union-Find':<35} {'O(E·α(N))':<18} {'O(E) — collect'}")
print(f"{'Fase 4 — Forward MLP':<35} {'O(N·W)':<18} {'O(W) — gradientes'}")
print(f"{'Fase 4 — L-BFGS iteraciones':<35} {'O(iter·N·W)':<18} {'O(W) por iteración'}")
print("-" * 62)
print(f"\nValores concretos del pipeline:")
print(f"  N total documentos  : {N_total:>10,}")
print(f"  |V| vocabulario     : {NUM_FEATURES:>10,}  (hashing trick)")
print(f"  K funciones MinHash : {K_MINHASH:>10}")
print(f"  p particiones Spark : {P_PARTICIONES:>10}")
print(f"  W parámetros MLP    : {W_MLP:>10,}")

## 8. Análisis de complejidad algorítmica y escalabilidad

Conexión con la unidad de **Teoría de la complejidad** del curso.

In [ ]:
import numpy as np

Ns = [1e4, 1e5, 5e5, 1e6, 1e7]

print("ESCALABILIDAD TEÓRICA")
print("=" * 70)
print(f"{'N docs':>12} {'Brute-force O(N²)':>22} {'LSH O(N·K)':>18} {'MLP O(N·W)':>16}")
print("-" * 70)
for n in Ns:
    bf  = n**2
    lsh = n * K_MINHASH
    mlp = n * W_MLP
    print(f"{n:>12,.0f} {bf:>22,.0f} {lsh:>18,.0f} {mlp:>16,.0f}")
print()
print("Conclusiones:")
print("  - LSH reduce la deduplicación de O(N²) a O(N·K) → tractable a escala")
print("  - El cuello de botella del pipeline es el shuffle del IDF: O(|V|·p)")
print("  - Spark distribuye O(N·W) del MLP entre p workers → speedup lineal")
print("  - Clase de complejidad: todos los algoritmos del pipeline son P")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

Ns_arr = np.logspace(3, 7, 100)
ax.plot(Ns_arr, Ns_arr**2,          label="Brute-force O(N²)",  linewidth=2)
ax.plot(Ns_arr, Ns_arr * K_MINHASH, label=f"LSH O(N·K), K={K_MINHASH}",   linewidth=2, linestyle="--")
ax.plot(Ns_arr, Ns_arr * W_MLP,     label=f"MLP O(N·W), W={W_MLP:,}",     linewidth=2, linestyle=":")
ax.axvline(x=N_total, color="gray", linestyle="-.", label=f"N real={N_total:,}")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("N (número de documentos)", fontsize=12)
ax.set_ylabel("Operaciones (escala log)", fontsize=12)
ax.set_title("Escalabilidad del pipeline", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()

fig.savefig(os.path.join(OUT_EVAL, "escalabilidad.png"), dpi=150)
plt.show()
print("Guardada en resultados/escalabilidad.png")

## 9. Resumen del pipeline completo

In [ ]:
print("RESUMEN DEL PIPELINE — Clasificador Masivo de Noticias")
print("=" * 60)
print(f"{'Fase':<40} {'Estado':<10} {'Resultado'}")
print("-" * 60)
print(f"{'Fase 1 — Ingesta':<40} {'OK':<10} {N_total:,} docs en Parquet")
print(f"{'Fase 2 — MapReduce + TF-IDF':<40} {'OK':<10} Vectores {NUM_FEATURES}-dim")

dedup_ok = os.path.exists(os.path.join(DEDUP_PATH, "corpus_deduplicado.parquet"))
print(f"{'Fase 3 — LSH Deduplicación':<40} {'OK' if dedup_ok else 'PENDIENTE':<10} Corpus limpio")
print(f"{'Fase 4 — Red Neuronal MLP':<40} {'OK':<10} Arquitectura {CAPAS}")
print(f"{'Fase 5 — Evaluación':<40} {'OK':<10} Acc={acc:.3f} F1={f1_w:.3f}")
print("-" * 60)
print(f"\nTemas del curso cubiertos:")
temas = [
    "Almacenamiento distribuido (HDFS/Parquet)",
    "Algoritmos MapReduce + modelo costo-comunicación",
    "TF-IDF e índice invertido",
    "Medidas de similitud (Jaccard, coseno)",
    "LSH con MinHash y Random Projection",
    "Redes neuronales MLP con PySpark MLlib",
    "Evaluación de modelos (F1, accuracy, precisión, recall)",
    "Teoría de la complejidad (P, escalabilidad)",
]
for t in temas:
    print(f"  ✓ {t}")

In [ ]:
spark.stop()
print("Pipeline completo. Resultados en la carpeta 'resultados/'.")